# GraphSAGE on PrimeKG — Colab T4

**Before running:** Runtime → Change runtime type → **T4 GPU**, and upload `pcgate_colab_bundle.zip`
to the top level of *My Drive*.

Results are written to `My Drive/pcgate/results/`. If the runtime disconnects, re-run cells 1–4 and
then continue; finished runs are skipped automatically, so nothing is duplicated or recomputed.

| section | what it does | time |
|---|---|---|
| 1–4 | set up, verify the pinned data | ~2 min |
| 5–8 | GraphSAGE on the **all-drug** pool (secondary protocol) | ~15 min |
| 9 | GraphSAGE on the **eligible-drug** pool (**primary** protocol) | ~15 min |
| 10 | ablation: no indication-derived features | ~13 min |

## 1. Check the GPU

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 2. Mount Drive and unpack the bundle

Re-run this after uploading a new bundle, otherwise the old code stays in `/content`.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!rm -rf /content/pcgate && mkdir -p /content/pcgate
!unzip -q /content/drive/MyDrive/pcgate_colab_bundle.zip -d /content/pcgate
%cd /content/pcgate
!ls

## 3. Install PyTorch Geometric

Colab already ships torch, pandas, scikit-learn and psutil.

In [ ]:
!pip install -q torch_geometric pyyaml
import torch, torch_geometric
print(torch.__version__, torch_geometric.__version__, 'cuda:', torch.cuda.is_available())

## 4. Verify the pinned data

The tests cover metrics, splits, pools and the leakage checks. Loading a split re-verifies its
stored pool hashes, and `run` re-checks integrity against the config actually being trained.

In [ ]:
!python -m pytest -q tests -p no:cacheprovider
!python -c "from pcgate.utils import read_json; [print(s, read_json(f'data/processed/primekg/splits/{s}_s0/integrity_checks.json')['all_passed']) for s in ('random','drug','disease')]"
!mkdir -p /content/drive/MyDrive/pcgate/results/primekg
!cp -rn results/primekg/* /content/drive/MyDrive/pcgate/results/primekg/ 2>/dev/null; echo done

## 5. Secondary protocol — GraphSAGE on the all-drug pool

3 settings × 3 seeds. Already-finished runs are skipped, so re-running this costs nothing.

In [ ]:
!python -m pcgate run --config configs/primekg_colab.yaml --only sage

## 6. Memory / runtime profile on the GPU

In [ ]:
!python -m pcgate profile --config configs/primekg_colab.yaml --models sage

## 7. Summaries, coverage, candidate table

In [ ]:
!python -m pcgate summarize  --config configs/primekg_colab.yaml
!python -m pcgate coverage   --config configs/primekg_colab.yaml
!python -m pcgate candidates --config configs/primekg_colab.yaml --model sage | head -20

## 8. PRIMARY protocol — eligible-drug pool

Sections 5–7 ranked against all 7,957 drug-type nodes. Only 1,801 are ever indicated for anything,
so ~78 % of the unlabeled pairs were trivially rejectable and an *eligibility oracle* that ignores
the disease entirely reached **AUROC 0.89**. AP and AUROC were largely measuring that; Recall@10,
Precision@10 and NDCG@10 were never affected.

The pools are now the 1,801 eligible drugs. **The train/val/test pairs are identical** (verified by
hash), so this is purely an evaluation change — but validation pools changed too, so early stopping
may land on different checkpoints. Splits for both protocols ship in the bundle; no `prepare` needed.

Expect popularity to drop to AP ≈ 0.36 / AUROC ≈ 0.72, and the oracle to fall back to chance.

In [ ]:
!python -m pcgate run       --config configs/primekg_colab_eligible.yaml --only popularity sage
!python -m pcgate pooldiag  --config configs/primekg_colab_eligible.yaml
!python -m pcgate summarize --config configs/primekg_colab_eligible.yaml
!python -m pcgate coverage  --config configs/primekg_colab_eligible.yaml

### Pool comparison: all drugs vs eligible drugs, same models

In [ ]:
import pandas as pd
old = pd.read_csv('/content/drive/MyDrive/pcgate/results/primekg/all_runs.csv').assign(pool='all drugs (7957)')
new = pd.read_csv('/content/drive/MyDrive/pcgate/results/primekg_eligible/all_runs.csv').assign(pool='eligible (1801)')
cmp = pd.concat([old, new]).groupby(['setting', 'model', 'pool'])[['ap', 'auroc', 'recall@10', 'ndcg@10']].mean()
print(cmp.round(4).to_string())
print('\nRanking metrics should be ~unchanged; AP and AUROC should fall.')

## 9. Ablation — no indication-derived node features

Every node carries a count of its *training* indication edges. That column is 0 for each held-out
disease and large for each training disease, so a model can learn "this disease already has known
treatments" instead of learning from biology — a plausible cause of the disease-held-out collapse
(GraphSAGE Recall@10 0.053 vs popularity 0.082; 0.016 on the lowest-degree tertile).

The feature is leakage-free (training edges only) and all six integrity checks pass either way, so
this is a design question, not a bug. The ablation runs under the **primary** protocol with the
feature off (103 → 99 features) and changes nothing else.

The integrity line below should read **99 features, 0 indication-derived**. Run section 8 first —
the comparison reads its results.

In [ ]:
!python -m pcgate run       --config configs/primekg_colab_noindfeat.yaml --only sage
!python -m pcgate summarize --config configs/primekg_colab_noindfeat.yaml
!python -m pcgate coverage  --config configs/primekg_colab_noindfeat.yaml

### Feature comparison: with vs without the indication features

In [ ]:
import pandas as pd
with_f = pd.read_csv('/content/drive/MyDrive/pcgate/results/primekg_eligible/all_runs.csv').assign(features='with (103)')
without = pd.read_csv('/content/drive/MyDrive/pcgate/results/primekg_eligible_noindfeat/all_runs.csv').assign(features='without (99)')
cmp = pd.concat([with_f, without])
cmp = cmp[cmp.model == 'sage'].groupby(['setting', 'features'])[['ap', 'recall@10', 'auroc']].agg(['mean', 'std'])
print(cmp.round(4).to_string())

## 10. Download every result

In [ ]:
!cd /content/drive/MyDrive/pcgate/results && zip -qr /content/primekg_results.zip . -x '*.pt'
from google.colab import files
files.download('/content/primekg_results.zip')